# 06 · Series de tiempo y anomalías

Una serie temporal agrega orden, dependencia y riesgo de mirar el futuro. Este lab cubre tendencia, estacionalidad, lags, rolling features, validación temporal, baseline de forecasting y varios enfoques de anomalías.

## Objetivos
- Distinguir tendencia, estacionalidad, ciclos, ruido y shocks.
- Crear lags y ventanas móviles sin leakage.
- Usar `TimeSeriesSplit` en lugar de CV aleatoria.
- Construir baselines naive y ML-based forecasting.
- Comparar z-score robusto, Isolation Forest y Local Outlier Factor.
- Entender anomalías puntuales, contextuales y colectivas.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
SEED=42
rng=np.random.default_rng(SEED); n=730; t=np.arange(n)
trend=.015*t; weekly=1.5*np.sin(2*np.pi*t/7); monthly=2*np.sin(2*np.pi*t/30)
y=20+trend+weekly+monthly+rng.normal(0,.8,n)
y[[120,360,600]] += [9,-11,12]
df=pd.DataFrame({'date':pd.date_range('2024-01-01',periods=n,freq='D'),'y':y}).set_index('date')
df.plot(figsize=(14,4),title='Serie sintética con tendencia, estacionalidad y shocks'); plt.show()

## 1. Baselines de forecasting
Antes de ARIMA, Prophet, XGBoost o redes neuronales, compara contra:
- **naive:** mañana = hoy;
- **seasonal naive:** mañana = valor de la misma posición del ciclo anterior;
- media/mediana móvil.

Un modelo sofisticado que no supera seasonal-naive no está aportando valor.


In [ ]:
test=df.iloc[-120:].copy(); train=df.iloc[:-120].copy()
# baseline 1: lag 1
pred_naive=df.y.shift(1).loc[test.index]
# baseline 2: semana anterior
pred_week=df.y.shift(7).loc[test.index]
for name,p in [('naive_lag1',pred_naive),('seasonal_lag7',pred_week)]:
    print(name,'MAE=',round(mean_absolute_error(test.y,p),3),'RMSE=',round(mean_squared_error(test.y,p)**.5,3))

## 2. Feature engineering temporal
Las features deben usar **solo pasado**. Un `rolling(7).mean()` aplicado sin `shift(1)` contiene el valor actual y puede producir leakage si el objetivo es predecir ese mismo instante.


In [ ]:
work=df.copy()
for lag in [1,2,7,14,28]: work[f'lag_{lag}']=work.y.shift(lag)
work['roll7_mean']=work.y.shift(1).rolling(7).mean(); work['roll7_std']=work.y.shift(1).rolling(7).std()
work['roll30_mean']=work.y.shift(1).rolling(30).mean()
work['dow']=work.index.dayofweek; work['month']=work.index.month
work=work.dropna(); work.head()

## 3. Modelo ML para forecasting tabular
Un Random Forest o gradient boosting puede modelar lags, calendario y rolling statistics. Para horizontes múltiples hay estrategias: recursive, direct, multi-output y seq2seq.


In [ ]:
features=[c for c in work.columns if c!='y']; split=len(work)-120
tr,te=work.iloc[:split],work.iloc[split:]
rf=RandomForestRegressor(n_estimators=500,min_samples_leaf=3,random_state=SEED,n_jobs=-1).fit(tr[features],tr.y)
p=rf.predict(te[features])
print('RF MAE',mean_absolute_error(te.y,p),'RMSE',mean_squared_error(te.y,p)**.5)
plt.figure(figsize=(14,4)); plt.plot(te.index,te.y,label='real'); plt.plot(te.index,p,label='RF'); plt.legend(); plt.show()

## 4. Validación temporal
Nunca mezcles futuro dentro de train. `TimeSeriesSplit` expande el entrenamiento de izquierda a derecha. Para datos con múltiples entidades hay que combinar lógica de grupos y tiempo.


In [ ]:
tscv=TimeSeriesSplit(n_splits=5); scores=[]
Xw,yw=work[features],work.y
for fold,(itr,iva) in enumerate(tscv.split(Xw),1):
    m=RandomForestRegressor(n_estimators=200,min_samples_leaf=3,random_state=SEED,n_jobs=-1).fit(Xw.iloc[itr],yw.iloc[itr])
    pred=m.predict(Xw.iloc[iva]); scores.append(mean_absolute_error(yw.iloc[iva],pred)); print('fold',fold,'MAE',round(scores[-1],3))
print('media',np.mean(scores),'std',np.std(scores))

## 5. Anomalías: no existe una única definición
- **Puntual:** un valor extremo.
- **Contextual:** normal globalmente, raro para ese horario/segmento.
- **Colectiva:** una secuencia completa es anómala aunque cada punto aislado parezca normal.

En operación real conviene detectar anomalías sobre **residuos de un modelo esperado**, no solo sobre el valor crudo, porque así quitamos tendencia/estacionalidad.


In [ ]:
# residuo respecto de una media móvil estacional simple
base=df.y.shift(7); resid=(df.y-base).dropna(); med=resid.median(); mad=np.median(np.abs(resid-med)); robust_z=.6745*(resid-med)/(mad+1e-9)
anom_robust=robust_z.abs()>3.5
iso=IsolationForest(contamination=.01,random_state=SEED).fit(resid.to_numpy().reshape(-1,1)); anom_iso=iso.predict(resid.to_numpy().reshape(-1,1))==-1
lof=LocalOutlierFactor(n_neighbors=25,contamination=.01); anom_lof=lof.fit_predict(resid.to_numpy().reshape(-1,1))==-1
print('robust',anom_robust.sum(),'isolation',anom_iso.sum(),'lof',anom_lof.sum())

In [ ]:
plt.figure(figsize=(14,4)); plt.plot(df.index,df.y,label='serie')
idx=resid.index[anom_iso]; plt.scatter(idx,df.loc[idx,'y'],marker='x',s=80,label='IsolationForest')
plt.legend(); plt.title('Anomalías detectadas sobre residuo estacional'); plt.show()

## 6. Concept drift y cambio estructural
Una anomalía no es lo mismo que drift. Drift implica que cambia la distribución o la relación $P(y|x)$ a lo largo del tiempo. Un cambio de política, sensor, población, proceso administrativo o definición de variable puede degradar un modelo sin producir puntos extremos. Lo veremos en MLOps.

## Extensiones importantes
- ARIMA/SARIMA y modelos de estado;
- exponential smoothing;
- Prophet;
- XGBoost/LightGBM con lags;
- LSTM/GRU;
- Temporal Convolutional Networks;
- N-BEATS, DeepAR, Temporal Fusion Transformer;
- change-point detection y CUSUM.

## Ejercicios
1. Implementa seasonal-naive con periodo 30.
2. Añade sin/cos para `dayofweek` y compara contra enteros.
3. Prueba `HistGradientBoostingRegressor`.
4. Simula un cambio de nivel permanente y distingue shock vs drift.
5. Detecta anomalías sobre residuos del forecasting en lugar de `y`.
6. Evalúa forecasting con rolling-origin backtesting.
7. Explica por qué MAPE falla cuando $y$ está cerca de cero.
